# Home Values — Why a Median Cannot Be Averaged

ACS table `B25077` publishes a median home value for every county. The obvious thing to do with
3,216 county medians is average them. That gives **209,224** against a published national median
of **303,400** — **31.0% low**.

The obvious fix is to weight by owner-occupied units. That gives **349,878** — **15.3% high**,
and high in **44 of 52** states. Weighting repairs the magnitude and not the error, because the
error is not one of weighting: **a median of a pooled population is not any function of the
medians of its parts.**

The correct method is to pool the *distribution* — table `B25075`, which publishes 26 value
brackets per county — and interpolate. Done that way, this notebook reproduces the Bureau's own
published state median to within **0.0260%** in **52 of 52** states.

At the national level the same method first returns **300,394**, **0.99% low**, and the reason is
not the method. `county:*` returns **3,222** rows and **78** of them are Puerto Rico municipios;
`us:*` excludes Puerto Rico. The two geography levels of one API do not cover the same universe,
and that difference alone — **844,228** owner-occupied units — moves the answer by a full
percentage point. Removing it gives **303,438**, **0.01%** from the published figure.

Nothing raises at any point in that sequence. Every number above is plausible.

## Learning Objectives

1. **Know which published statistics are aggregable and which are not.** Counts add. Medians do
   not, under any weighting.
2. **Reconstruct a median from a pooled bracket distribution**, and validate the reconstruction
   against the publisher's own figure.
3. **Check that two geography levels of one API cover the same universe** before comparing an
   aggregate of one against the other.
4. **Notice the geographies where the statistic does not exist**, rather than dropping them.

## Prerequisites

- D27 (`D27_rent_burden`) — bracket interpolation within one geography, with margins. This
  notebook uses the same machinery across geographies, which is the harder case.
- D01 — Puerto Rico as a unit that is reported with the states and is not one. Here the issue is
  not leverage but universe.
- Handbook §3 (denominators, universes and partitions).

## Data Sources

**Census ACS 5-year, 2023.** `B25077` (median value, owner-occupied units) and `B25075` (value,
26 brackets) at county, state and national level, plus `B25003` for owner-occupied unit counts.

**Vintage is pinned.** ACS 5-year estimates are revised and re-based; `year=2023` is passed
explicitly and there is no "latest" in this client.

## 1. Setup

In [1]:
"""Configuration. Bracket edges are declared from the published labels, not inferred."""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

_here = Path.cwd().resolve()
for _cand in (_here, *_here.parents):
    if (_cand / "src" / "krl_tutorials_data").is_dir():
        sys.path.insert(0, str(_cand / "src")); break
else:
    raise SystemExit("Could not locate krl_tutorials_data from " + str(_here))

from krl_tutorials_data.sources import fetch_acs
from krl_tutorials_data.brackets import BracketEdges, median_from_brackets
from krl_tutorials_data.tabulation import check_partition

#: Pinned vintage. There is no "latest".
YEAR = 2023

MEDIAN, MEDIAN_MOE = "B25077_001E", "B25077_001M"
OWNER_UNITS = "B25003_002E"

#: B25075 lower edges, transcribed from the published variable labels
#: ("Less than $10,000", "$10,000 to $14,999", ... "$2,000,000 or more").
#: The top bracket is open, which `BracketEdges` records as inf.
_LOWER = [0, 10_000, 15_000, 20_000, 25_000, 30_000, 35_000, 40_000, 50_000,
          60_000, 70_000, 80_000, 90_000, 100_000, 125_000, 150_000, 175_000,
          200_000, 250_000, 300_000, 400_000, 500_000, 750_000, 1_000_000,
          1_500_000, 2_000_000]
VALUE_BRACKETS = BracketEdges(
    lower=tuple(float(x) for x in _LOWER),
    upper=tuple(float(x) for x in _LOWER[1:] + [float("inf")]),
    units="dollars",
)
BRACKET_CODES = [f"B25075_{i:03d}E" for i in range(2, 28)]

assert len(BRACKET_CODES) == VALUE_BRACKETS.n, "one code per bracket"
print(f"vintage        : ACS 5-year {YEAR}")
print(f"brackets       : {VALUE_BRACKETS.n}, top open = {VALUE_BRACKETS.top_is_open}")
print(f"bracket codes  : {BRACKET_CODES[0]} .. {BRACKET_CODES[-1]}")


vintage        : ACS 5-year 2023
brackets       : 26, top open = True
bracket codes  : B25075_002E .. B25075_027E


## 2. Retrieval

In [2]:
"""Three geography levels of one table, plus the distribution behind it."""
_summary = {MEDIAN: "Estimate!!Median value (dollars)",
            MEDIAN_MOE: "Margin of Error!!Median value (dollars)",
            OWNER_UNITS: "Estimate!!Total:!!Owner occupied"}
_dist = {code: f"Estimate!!Total:!!bracket {i}" for i, code in enumerate(BRACKET_CODES, start=1)}

counties, prov_cty = fetch_acs(year=YEAR, variables=_summary, geography="county:*")
states, _ = fetch_acs(year=YEAR, variables=_summary, geography="state:*")
nation, _ = fetch_acs(year=YEAR, variables=_summary, geography="us:*")
county_dist, _ = fetch_acs(year=YEAR, variables=_dist, geography="county:*")
nation_dist, _ = fetch_acs(year=YEAR, variables=_dist, geography="us:*")

for frame, cols in ((counties, _summary), (states, _summary), (nation, _summary),
                    (county_dist, _dist), (nation_dist, _dist)):
    for code in cols:
        frame[code] = pd.to_numeric(frame[code], errors="coerce")

print(f"county rows      : {len(counties):,}")
print(f"state rows       : {len(states):,}")
print(f"national rows    : {len(nation):,}")
print(f"published US median : {nation[MEDIAN].iloc[0]:,.0f} "
      f"(MOE {nation[MEDIAN_MOE].iloc[0]:,.0f})")


county rows      : 3,222
state rows       : 52
national rows    : 1
published US median : 303,400 (MOE 362)


## 3. The counties where the statistic does not exist

In [3]:
"""Before any aggregation: where is there no median at all?"""
missing = counties[counties[MEDIAN].isna()]
print(f"counties with no published median: {len(missing)}")
print(missing[["NAME", OWNER_UNITS]].to_string(index=False))
print()
print(f"owner-occupied units in them : {int(missing[OWNER_UNITS].sum()):,}")
print(f"owner-occupied units in total: {int(counties[OWNER_UNITS].sum()):,}")
print(f"share                        : "
      f"{missing[OWNER_UNITS].sum() / counties[OWNER_UNITS].sum():.6%}")

usable = counties.dropna(subset=[MEDIAN]).copy()
print()
print(f"counties carrying a median   : {len(usable):,}")


counties with no published median: 6
                  NAME  B25003_002E
Kalawao County, Hawaii          0.0
 Eureka County, Nevada        538.0
  Kenedy County, Texas          6.0
    King County, Texas         38.0
  Loving County, Texas          4.0
    Real County, Texas        772.0

owner-occupied units in them : 1,358
owner-occupied units in total: 83,736,265
share                        : 0.001622%

counties carrying a median   : 3,216


Six counties have no median value. One of them — Kalawao County, Hawaii — has **zero**
owner-occupied units, so no median could exist. The other five have between **4** and **772**,
which is the Bureau declining to publish rather than an absence of housing.

Together they hold **1,358** owner-occupied units: **0.001622%** of the national total. Dropping
them costs nothing here, and that is exactly why it is worth doing deliberately. `dropna()` would
have removed them silently, and the same call on a table where the suppressed geographies were
the ones that mattered would have removed those just as quietly.

**[rule]** Count what a `dropna` removes and what it is worth before calling it, not after.

## 4. Three ways to average a median, all wrong

In [4]:
"""The three things an analyst actually does with a column of medians."""
published = float(nation[MEDIAN].iloc[0])

unweighted_mean = usable[MEDIAN].mean()
unweighted_median = usable[MEDIAN].median()
weighted_mean = float(np.average(usable[MEDIAN], weights=usable[OWNER_UNITS]))

print(f"published national median         : {published:>10,.0f}")
print()
for label, value in (("mean of county medians", unweighted_mean),
                     ("median of county medians", unweighted_median),
                     ("owner-unit-weighted mean", weighted_mean)):
    print(f"{label:<34}: {value:>10,.0f}   {(value - published) / published:+7.1%}")


published national median         :    303,400

mean of county medians            :    209,224    -31.0%
median of county medians          :    172,300    -43.2%
owner-unit-weighted mean          :    349,878    +15.3%


**31.0% low. 43.2% low. 15.3% high.**

The first two fail because a county is one observation regardless of whether it holds 4 owner-occupied
units or 400,000, and most counties are small and cheap. The weighted mean corrects that and is still
wrong by **15.3%**, in the other direction.

That last one is the dangerous result, because it looks like the fix. It is wrong for a reason
no amount of weighting addresses: a mean of medians is a mean of order statistics, and averaging
order statistics does not produce an order statistic of the pooled distribution. With a
right-skewed variable the county medians sit below their own county means, but the pooled median
sits below the *unit-weighted* average of those medians — and nothing constrains the size of the
gap.

**[rule]** Medians, percentiles, ratios of medians and indices built from them are not
aggregable. If the published statistic is not a count or a total, no weighting scheme recovers
the aggregate from the parts.

## 5. Is the bias one-signed? Fifty-two states

In [5]:
"""Per-state, so the direction can be read rather than asserted."""
state_median = states.set_index("state")[MEDIAN]
rows = []
for code, group in usable.groupby("state"):
    if code not in state_median.index or pd.isna(state_median.loc[code]):
        continue
    pub = float(state_median.loc[code])
    rows.append({
        "state": states.set_index("state").loc[code, "NAME"],
        "published": pub,
        "mean_err": (group[MEDIAN].mean() - pub) / pub * 100,
        "wmean_err": (np.average(group[MEDIAN], weights=group[OWNER_UNITS]) - pub) / pub * 100,
        "counties": len(group),
    })
per_state = pd.DataFrame(rows)

print(f"states compared: {len(per_state)}")
print()
for col, label in (("mean_err", "unweighted mean"), ("wmean_err", "owner-weighted mean")):
    err = per_state[col]
    print(f"{label:<22} median {err.median():+7.2f}%   "
          f"min {err.min():+7.2f}%   max {err.max():+7.2f}%   "
          f"high in {int((err > 0).sum())}/{len(err)}")
print()
print("largest weighted-mean errors:")
print(per_state.reindex(per_state.wmean_err.abs().sort_values(ascending=False).index)
      .head(5)[["state", "published", "wmean_err", "counties"]]
      .to_string(index=False, float_format=lambda v: f"{v:,.2f}"))


states compared: 52

unweighted mean        median  -20.79%   min  -41.66%   max  +17.71%   high in 4/52
owner-weighted mean    median   +2.03%   min   -1.82%   max  +16.73%   high in 44/52

largest weighted-mean errors:
      state  published  wmean_err  counties
   New York 403,000.00      16.73        62
    Wyoming 285,100.00      12.99        23
   Virginia 360,700.00      10.25       133
 California 695,400.00       7.96        58
Connecticut 343,200.00       6.80         9


The weighted mean is high in **44 of 52** states, median error **+2.03%**, and it reaches
**+16.73%** in New York. New York is instructive because it is the case weighting should have
handled: **62** counties, a handful of which hold most of the state's owner-occupied units and
carry medians far above the rest. Weighting gives those counties the influence their unit counts
deserve and still misses the state median by a sixth, because their *medians* are not the values
the pooled distribution is made of.

The bias is one-signed and its size is not predictable from anything in the table. That is a
worse property than a large random error: a one-signed error survives averaging across states,
and looks like a finding.

## 6. The method that works — pool the distribution

In [6]:
"""Sum the bracket counts across a state's counties, then interpolate."""
state_dist_rows = []
for code, group in county_dist.groupby("state"):
    if code not in state_median.index or pd.isna(state_median.loc[code]):
        continue
    pooled = group[BRACKET_CODES].sum().to_numpy(dtype=float)
    pub = float(state_median.loc[code])
    estimate = median_from_brackets(pooled, VALUE_BRACKETS)
    state_dist_rows.append({
        "state": states.set_index("state").loc[code, "NAME"],
        "published": pub,
        "pooled": estimate,
        "err_pct": (estimate - pub) / pub * 100,
        "units": int(pooled.sum()),
    })
pooled_states = pd.DataFrame(state_dist_rows)

print(f"states reconstructed : {len(pooled_states)}")
print(f"median error         : {pooled_states.err_pct.median():+.4f}%")
print(f"worst error          : {pooled_states.err_pct.abs().max():.4f}%")
print(f"within 0.05%         : {int((pooled_states.err_pct.abs() <= 0.05).sum())} "
      f"of {len(pooled_states)}")
print()
print("largest residuals:")
print(pooled_states.reindex(pooled_states.err_pct.abs().sort_values(ascending=False).index)
      .head(5).to_string(index=False, float_format=lambda v: f"{v:,.4f}"))


states reconstructed : 52
median error         : +0.0014%
worst error          : 0.0260%
within 0.05%         : 52 of 52

largest residuals:
        state    published       pooled  err_pct   units
West Virginia 155,600.0000 155,640.4933   0.0260  536082
  Mississippi 161,400.0000 161,359.5129  -0.0251  786289
      Indiana 201,600.0000 201,551.8795  -0.0239 1886485
         Ohio 199,200.0000 199,246.3648   0.0233 3235568
      Alabama 195,100.0000 195,056.1930  -0.0225 1377062


**52 of 52 within 0.05%**, worst residual **0.0260%**.

That agreement is not a coincidence and it is not this notebook being clever. Linear
interpolation inside the containing bracket reproduces the Bureau's published median because it
is the same operation the Bureau performs on the same grouped distribution. The value of running
it here is not the estimate — the estimate is already published — it is that **the method is now
validated against an external authority**, so it can be trusted at geographies where no median is
published at all.

**[rule]** When a helper reconstructs a published figure, check it against that figure before
using it where nothing is published. A reconstruction that has never been compared to a known
answer is an assumption with arithmetic in it.

## 7. The same method, nationally — and a universe that does not match

In [7]:
"""Pool every county in the country. Then check the universe."""
pooled_national = county_dist[BRACKET_CODES].sum().to_numpy(dtype=float)
published_national = nation_dist[BRACKET_CODES].iloc[0].to_numpy(dtype=float)

report = check_partition(pooled_national, published_national.sum(),
                         label="county bracket counts vs published US total")
print(report.render())
print()
print(f"pooled median over all counties : "
      f"{median_from_brackets(pooled_national, VALUE_BRACKETS):,.0f}   "
      f"{(median_from_brackets(pooled_national, VALUE_BRACKETS) - published) / published:+.2%}")


county bracket counts vs published US total: sum of parts 83,736,265 vs published total 82,892,037  (ratio 1.0102, residual 844,228) -> OVER

pooled median over all counties : 300,394   -0.99%


`OVER` by **844,228** units — and the pooled median comes out at **300,394**, **0.99%** below
the published **303,400**.

The method did not fail. The universe did. Nothing in the API says so, both requests succeeded,
and a `0.99%` discrepancy is small enough to be attributed to rounding by anyone not checking.

In [8]:
"""Which rows are the difference?"""
PUERTO_RICO = "72"
pr = county_dist[county_dist["state"] == PUERTO_RICO]
print(f"county:* rows                    : {len(county_dist):,}")
print(f"of which Puerto Rico municipios  : {len(pr)}")
print(f"owner-occupied units in them     : {int(pr[BRACKET_CODES].sum().sum()):,}")
print(f"check_partition residual above   : {report.residual:,.0f}")
print()

mainland = county_dist[county_dist["state"] != PUERTO_RICO]
pooled_mainland = mainland[BRACKET_CODES].sum().to_numpy(dtype=float)
check = check_partition(pooled_mainland, published_national.sum(),
                        label="county bracket counts, ex-PR, vs published US total")
print(check.render())
print()
for label, counts in (("pooled ex-Puerto Rico", pooled_mainland),
                      ("published US brackets", published_national)):
    est = median_from_brackets(counts, VALUE_BRACKETS)
    print(f"{label:<24}: {est:>10,.0f}   {(est - published) / published:+.2%}")
print(f"{'published B25077':<24}: {published:>10,.0f}")


county:* rows                    : 3,222
of which Puerto Rico municipios  : 78
owner-occupied units in them     : 844,228
check_partition residual above   : 844,228

county bracket counts, ex-PR, vs published US total: sum of parts 82,892,037 vs published total 82,892,037  (ratio 1.0000, residual 0) -> EXACT

pooled ex-Puerto Rico   :    303,438   +0.01%
published US brackets   :    303,438   +0.01%
published B25077        :    303,400


The residual is Puerto Rico exactly: **78** municipios, **844,228** owner-occupied units, and
`check_partition` now reports **EXACT**.

`county:*` includes Puerto Rico. `us:*` does not. Two geography levels of the same table in the
same API, and their universes differ — so an aggregate built from one and compared against the
other is comparing two different countries.

Excluding it, the pooled median is **303,438** against a published **303,400**: **+0.01%**. And
interpolating the Bureau's *own* national brackets gives the identical **303,438**, which
confirms the residual is the interpolation's rounding and not the pooling.

**[rule]** Before comparing an aggregate of sub-geographies against a published total, run the
partition check. Two levels of one API are not guaranteed to cover one universe, and the failure
is silent in both directions.

## 8. What the county medians can and cannot support

In [9]:
"""The margins, which the aggregation questions above never touched."""
rel = usable[MEDIAN_MOE] / usable[MEDIAN] * 100
print(f"relative MOE on county medians   : median {rel.median():.2f}%   "
      f"p90 {rel.quantile(0.90):.2f}%   max {rel.max():.2f}%")
print(f"counties with MOE above 10%      : {int((rel > 10).sum()):,} of {len(usable):,}")
print()
nat_rel = nation[MEDIAN_MOE].iloc[0] / published * 100
print(f"relative MOE on the US median    : {nat_rel:.3f}%")
print()
print("The pooled-bracket estimate in section 6 has no margin of error at all.")
print("Bracket counts carry their own MOEs, and combining them does not give the")
print("MOE of an interpolated percentile -- that needs the replicate weights.")


relative MOE on county medians   : median 5.88%   p90 15.08%   max 66.87%
counties with MOE above 10%      : 775 of 3,216

relative MOE on the US median    : 0.119%

The pooled-bracket estimate in section 6 has no margin of error at all.
Bracket counts carry their own MOEs, and combining them does not give the
MOE of an interpolated percentile -- that needs the replicate weights.


Median relative margin **5.88%**; **775** of **3,216** counties carry a margin above **10%** of
their own estimate; the worst is **66.87%**. Against that, the national median's margin is
**0.119%**.

So the county medians are individually imprecise and the reconstructed aggregate is precise but
unquantified. Those are different deficiencies and neither substitutes for the other: section 6's
estimate is right and cannot be given a confidence interval by any method available in this
table, while the county figures have intervals and are too wide to rank.

**[rule]** State which of the two you have. A point estimate validated against a publisher is not
the same object as an estimate with a margin, and a reader cannot tell them apart from the
number.

## Limitations

1. **One vintage.** Everything here is ACS 5-year 2023. The bracket edges are nominal dollars and
   are not re-based between vintages, so bracket counts across years are not comparable without
   deflation and the `$2,000,000 or more` bracket fills over time.

2. **No margin on the reconstruction.** Section 6's method reproduces the published median and
   cannot produce a confidence interval for it. The Bureau computes median MOEs from replicate
   weights, which this table does not carry.

3. **The top bracket is open.** `median_from_brackets` raises on an unidentified percentile. No
   geography here has its median inside the open bracket, but a high-value tract-level analysis
   could, and this notebook does not exercise that path.

4. **The interpolation assumption is uniformity within a bracket.** It agrees with the Bureau
   because the Bureau assumes the same thing, not because either is true of the housing stock.

5. **Puerto Rico is the universe difference here, not the only possible one.** Other tables and
   other geography levels differ in other ways; the partition check is the general defence, and
   the specific finding is not.

6. **Six counties are excluded throughout**, holding 0.001622% of owner-occupied units. Section 3
   states what that costs. On a different variable the suppressed geographies could be the
   substantive ones.

7. **This says nothing about housing affordability.** Median value is a stock price level, not a
   burden, and it is not comparable across places without local income.

## Key Takeaways

1. **Medians are not aggregable.** Mean of county medians: **31.0% low**. Median of them:
   **43.2% low**. Owner-unit-weighted mean: **15.3% high**.

2. **Weighting is not the fix.** The weighted mean is high in **44 of 52** states, median error
   **+2.03%**, worst **+16.73%** — one-signed, so it survives averaging and looks like a finding.

3. **Pool the distribution instead.** Summing `B25075` bracket counts and interpolating
   reproduces the published state median within **0.05%** in **52 of 52** states.

4. **Validate a reconstruction against a published figure** before trusting it where nothing is
   published.

5. **`county:*` includes Puerto Rico; `us:*` does not.** **78** municipios, **844,228** owner
   units, a **0.99%** error in the national median, and no exception anywhere.

6. **`check_partition` is the defence** and it is one line. It reported `OVER` by exactly the
   Puerto Rico total, and `EXACT` once removed.

7. **Six counties publish no median**, one because it has no owner-occupied units at all.

8. **775 of 3,216 county medians carry a margin above 10%** of their own estimate. The
   reconstructed aggregate carries none at all. Say which you have.

## Exercises

1. **Show the aggregation is hopeless directly.** Construct two counties with identical medians
   and different distributions whose pooled median differs. Then state in one sentence why no
   weighting scheme could have distinguished them.

2. **Rebuild at a finer geography.** Repeat section 6 for census tracts within one state and
   compare against the published county medians. Report where the method degrades and why.

3. **Find another asymmetric universe.** Fetch a second ACS table at `county:*` and `us:*` and run
   `check_partition`. Report whether the Puerto Rico gap is a property of this table or of the
   API's geography levels.

4. **Bracket the open top.** For each state, compute how many owner-occupied units sit in
   `$2,000,000 or more` and identify any geography whose median would fall inside it. Say what
   `median_from_brackets` does there and why raising is correct.

5. **Deflate and compare.** Take `B25075` for 2013 and 2023, deflate the bracket edges, and say
   what can and cannot be concluded about the change in the distribution.

6. **Put a margin on it.** Research how the Bureau computes the MOE for a published median, and
   write two sentences on why the bracket MOEs in this table are not sufficient.

## References

- U.S. Census Bureau. *American Community Survey 5-Year Estimates*, 2023 — tables `B25077`
  (median value), `B25075` (value), `B25003` (tenure).
  https://api.census.gov/data/2023/acs/acs5
- U.S. Census Bureau. *American Community Survey Design and Methodology* — estimation, medians
  and margins of error. https://www.census.gov/programs-surveys/acs/methodology/design-and-methodology.html
- U.S. Census Bureau. *Geography Program — Puerto Rico.*
  https://www.census.gov/programs-surveys/geography.html
- Handbook §3 (denominators, universes and partitions), §14 (a second column crossing the first).
- D27 (`08-culture-society/D27_housing_affordability_and_gentrification/D27_rent_burden`) —
  bracket interpolation with margins, within one geography.
- D01 (`01-fundamentals/D01_income_and_poverty`) — Puerto Rico reported beside the states.
- `src/krl_tutorials_data/brackets.py` (`BracketEdges`, `median_from_brackets`) and
  `src/krl_tutorials_data/tabulation.py` (`check_partition`).